# Forests: many trees vote
**Machine Learning · Week 3 (Classification) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w3d5.html

A random forest grows many trees, each on a random resample of the data with random features at each
split, and lets them vote.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from matplotlib.colors import LinearSegmentedColormap

X, y = make_moons(n_samples=400, noise=0.3, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
models = {
    "one deep tree": DecisionTreeClassifier(random_state=0),
    "forest of 200 trees": RandomForestClassifier(n_estimators=200, random_state=0),
}
xx, yy = np.meshgrid(np.linspace(-2, 3, 250), np.linspace(-1.5, 2, 200))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, layout="constrained")
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    p = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=20, cmap=apricot_blue, alpha=0.5)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=np.where(y_train == 0, "#ff8a50", "#5577f5"), s=12)
    ax.set_title(f"{name}: test accuracy {model.score(X_test, y_test):.2f}")
plt.show()

## A free validation score: out-of-bag
Each tree's resample leaves out about a third of the examples. Scoring every example only with the trees that
never saw it gives an honest estimate without a separate validation set.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0).fit(X_train, y_train)
print(f"out-of-bag accuracy: {rf.oob_score_:.3f}")
print(f"test accuracy:       {rf.score(X_test, y_test):.3f}")

## Boosting, in one cell
A different way to combine trees: grow small trees **one after another**, each fixing the mistakes of the ones
before. On real tables of data, gradient boosting is often the strongest model of all
(on this small, noisy toy problem the forest happens to win).

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.1, random_state=0).fit(X_train, y_train)
print(f"gradient boosting test accuracy: {gb.score(X_test, y_test):.3f}")

## Your turn: how many trees are enough?
Train forests with 1, 3, 10, 30, 100 and 300 trees and print their test accuracy.

In [ ]:
for n in [1, 3, 10, 30, 100, 300]:
    acc = None   # TODO: fit RandomForestClassifier(n_estimators=n, random_state=0) and score it on the test set
    print(f"{n:3d} trees: test accuracy {acc}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for n in [1, 3, 10, 30, 100, 300]:
    acc = RandomForestClassifier(n_estimators=n, random_state=0).fit(X_train, y_train).score(X_test, y_test)
    print(f"{n:3d} trees: test accuracy {acc:.3f}")
print("Accuracy climbs quickly, then levels off: adding trees doesn't make a forest overfit, it just costs more time.")